# Predict Customer Chrun for Telecommunication company

### Library & Dataset import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set plot style
sns.set_style('whitegrid')

In [ ]:
!git clone "https://github.com/GeeksforgeeksDS/21-Days-21-Projects-Dataset"

In [ ]:
df = pd.read_csv('/content/21-Days-21-Projects-Dataset/Datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print("Dataset loaded successfully.")
print(f"Data shape: {df.shape}")
df.head()

## Data Cleaning

In [ ]:
df.info()

In [ ]:
print(f"Shape before cleaning: {df.shape}")

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"Shape after converting TotalCharges to numeric: {df.shape}")

In [ ]:
print(f"Number of missing TotalCharges: {df['TotalCharges'].isnull().sum()}")

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
print(f"Shape after dropping rows with missing Churn: {df.shape}")

In [ ]:
df.head()

In [ ]:
df['Churn'].value_counts()

## Baseline Performance (w/o feature engineering)

In [ ]:
X_base = df.drop('Churn', axis=1)
y_base = df['Churn']

In [ ]:
numerical_features_base = X_base.select_dtypes(include=np.number).columns.tolist()
categorical_features_base = X_base.select_dtypes(include=['object']).columns.tolist()

In [ ]:
preprocessor_base = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features_base),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_base)])

In [ ]:
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(X_base, y_base, test_size=0.2, random_state=42, stratify=y_base)

In [ ]:
baseline_model = Pipeline(steps=[('preprocessor', preprocessor_base),
                                 ('classifier', LogisticRegression(random_state=42, max_iter=1000))])

In [ ]:
baseline_model.fit(X_train_base, y_train_base)
y_pred_base = baseline_model.predict(X_test_base)

In [ ]:
print(classification_report(y_test_base, y_pred_base))

## Core Task - Feature Engineering

In [ ]:
df['tenure'].describe()

In [ ]:
df_eng = df.copy()
bins = [0, 12, 24, 48, 60, 73]
labels = ['0-1 Year', '1-2 Years', '2-4 Years', '4-5 Years', '5+ Years']
df_eng['tenure_group'] = pd.cut(df_eng['tenure'], bins=bins, labels=labels, right=False)

df_eng['MultipleLines'] = df_eng['MultipleLines'].replace({'No phone service': 'No'})
for col in ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']:
    df_eng[col] = df_eng[col].replace({'No internet service': 'No'})

df_eng['num_add_services'] = (df_eng[['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']] == 'Yes').sum(axis=1)
df_eng['monthly_charge_ratio'] = df_eng['MonthlyCharges'] / (df_eng['tenure'] + 1) # +1 to avoid division by zero
print("Feature engineering complete. New features added.")
df_eng.head()

### Additional Feture Engineering to increase Model Performance

### The combination of Contract and PaymentMethod could offer insights into customer retention.

### Combining the SeniorCitizen flag with other features might uncover specific patterns for elderly customers.

In [ ]:
df_eng['SeniorCitizen_Contract'] = df_eng['SeniorCitizen'].astype(str) + '_' + df_eng['Contract']
df_eng['SeniorCitizen_PaymentMethod'] = df_eng['SeniorCitizen'].astype(str) + '_' + df_eng['PaymentMethod']

### Customers with higher MonthlyCharges might have different behaviors.

In [ ]:
df_eng['High_Value_Customer'] = (df_eng['MonthlyCharges'] > df_eng['MonthlyCharges'].median()).astype(int)

### The combination of tenure_group and MonthlyCharges could give more information about how long a customer has been with the company and how much they are paying.

In [ ]:
df_eng['Tenure_MonthlyCharge_Interaction'] = df_eng['tenure_group'].astype(str) + '_' + (df_eng['MonthlyCharges'] > df_eng['MonthlyCharges'].median()).astype(str)

### Create a new feature that captures the churn rate based on the Contract type.

In [ ]:
churn_by_contract = df.groupby('Contract')['Churn'].mean()
df_eng['Churn_by_Contract'] = df_eng['Contract'].map(churn_by_contract)

### Customers with or without internet services might behave differently, especially in terms of churn.

In [ ]:
df_eng['No_Internet_Service'] = (df_eng['InternetService'] == 'No').astype(int)

### Feature that captures the ratio of total charges to monthly charges.

In [ ]:
df_eng['TotalCharges_to_MonthlyCharges'] = df_eng['TotalCharges'] / (df_eng['MonthlyCharges'] + 1)

### If customers have more than one service (e.g., PhoneService, MultipleLines, OnlineSecurity, etc.), they might be more loyal. Create a feature that captures the total number of services each customer subscribes to.

In [ ]:
df_eng['TotalServices'] = (df_eng[['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                                   'TechSupport', 'StreamingTV', 'StreamingMovies']] == 'Yes').sum(axis=1)

### The type of payment method (PaymentMethod) could impact churn.

In [ ]:
payment_method_counts = df['PaymentMethod'].value_counts()
df_eng['PaymentMethod_Ratio'] = df['PaymentMethod'].map(lambda x: payment_method_counts[x] / len(df))

### Some customers with families (i.e., dependents) may churn differently than those who are single.

In [ ]:
df_eng['Has_Partner_or_Dependents'] = ((df_eng['Partner'] == 'Yes') | (df_eng['Dependents'] == 'Yes')).astype(int)

### Customers who use multiple streaming services may have different churn patterns.

In [ ]:
df_eng['Num_Streaming_Services'] = (df_eng[['StreamingTV', 'StreamingMovies']] == 'Yes').sum(axis=1)

### Convert SeniorCitizen into more distinct age groups to better capture trends.

In [ ]:
df_eng['Age_Group'] = pd.cut(df_eng['SeniorCitizen'], bins=[-1, 0, 1], labels=['Non-Senior', 'Senior'])

### Customer Lifetime value

In [ ]:
df_eng['Customer_Lifetime_Value'] = df_eng['TotalCharges'] / (df_eng['tenure'] + 1)

In [ ]:
df_eng.head()

In [ ]:
df_eng.info()

## Model 2 - Performace with Engineered Features

In [ ]:
df_eng.drop('customerID', axis=1, inplace=True)

# Drop original tenure as we have a binned version now
df_eng.drop('tenure', axis=1, inplace=True)

# Define features (X) and target (y) for the engineered dataset
X_eng = df_eng.drop('Churn', axis=1)
y_eng = df_eng['Churn']

# Identify new feature types
numerical_features_eng = X_eng.select_dtypes(include=np.number).columns.tolist()
# Note: 'tenure_group' is now a categorical feature
categorical_features_eng = X_eng.select_dtypes(include=['object', 'category']).columns.tolist()

# Create the new preprocessing pipeline
preprocessor_eng = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features_eng),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_eng)])

# Split data
X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(X_eng, y_eng, test_size=0.2, random_state=42, stratify=y_eng)

# Create the full pipeline with the same classifier for a fair comparison
enhanced_model = Pipeline(steps=[('preprocessor', preprocessor_eng),
                                 ('classifier', LogisticRegression(random_state=42, max_iter=1000))])

# Train and evaluate the enhanced model
enhanced_model.fit(X_train_eng, y_train_eng)
y_pred_eng = enhanced_model.predict(X_test_eng)

print("--- Enhanced Model Performance (with Feature Engineering) ---")
print(classification_report(y_test_eng, y_pred_eng))

## Comparison & Conclusion

In [ ]:
# To get feature importance, let's quickly train a RandomForest model with the engineered data
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor_eng),
                               ('classifier', RandomForestClassifier(random_state=42))])
rf_pipeline.fit(X_train_eng, y_train_eng)

# Extract feature names after one-hot encoding
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = rf_pipeline.named_steps['classifier'].feature_importances_

feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False).head(15)

plt.figure(figsize=(12, 10))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='rocket', hue='Feature', legend=False)
plt.title('Top 15 Most Important Features (from Enhanced Model)')
plt.show()

## Feature Selection

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# Try different thresholds
thresholds = ['mean', 'median', 0.01]

for threshold in thresholds:
    selector = SelectFromModel(RandomForestClassifier(random_state=42), threshold=threshold)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_eng),
        ('selector', selector)
    ])
    pipeline.fit(X_train_eng, y_train_eng)

    X_train_sel = pipeline.transform(X_train_eng)
    X_test_sel = pipeline.transform(X_test_eng)

    print(f"\n[RandomForest] Threshold: {threshold}")
    print(f"Selected features: {X_train_sel.shape[1]}")

### RFE- Use RFE with a Logistic Regression or Random Forest estimator.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# Create a preprocessing + model pipeline for RFE
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# You can change n_features_to_select or use step='auto'
rfe_selector = RFE(estimator=log_reg, n_features_to_select=20, step=1)

# Combine with preprocessing
rfe_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_eng),
    ('feature_selection', rfe_selector)
])

rfe_pipeline.fit(X_train_eng, y_train_eng)

X_train_rfe = rfe_pipeline.transform(X_train_eng)
X_test_rfe = rfe_pipeline.transform(X_test_eng)

print(f"\n[RFE] Selected features: {X_train_rfe.shape[1]}")

### Filter Methods (Mutual Information & Chi-squared)

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif, chi2
from sklearn.preprocessing import MinMaxScaler

# First transform the data using the preprocessor
X_train_transformed = preprocessor_eng.fit_transform(X_train_eng)
X_test_transformed = preprocessor_eng.transform(X_test_eng)

# --- Mutual Information ---
mi_selector = SelectKBest(score_func=mutual_info_classif, k=30)
X_train_mi = mi_selector.fit_transform(X_train_transformed, y_train_eng)
X_test_mi = mi_selector.transform(X_test_transformed)

print(f"\n[Mutual Information] Selected features: {X_train_mi.shape[1]}")

# --- Chi-squared ---
# Chi2 requires non-negative values; use MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_transformed)
X_test_scaled = scaler.transform(X_test_transformed)

chi2_selector = SelectKBest(score_func=chi2, k=30)
X_train_chi2 = chi2_selector.fit_transform(X_train_scaled, y_train_eng)
X_test_chi2 = chi2_selector.transform(X_test_scaled)

print(f"[Chi-squared] Selected features: {X_train_chi2.shape[1]}")

### Consistency Analysis

In [ ]:
feature_mask = selector.get_support()
feature_names = preprocessor_eng.get_feature_names_out()
selected_features = feature_names[feature_mask]
print(selected_features)

In [ ]:
# Create the full pipeline with the preprocessor and the classifier
selected_features_model = Pipeline(steps=[('preprocessor', preprocessor_eng),
                                         ('classifier', LogisticRegression(random_state=42, max_iter=1000))])

# Train the model using the selected features
selected_features_model.fit(X_train_eng, y_train_eng)

In [ ]:
# Predict on the test set with selected features
y_pred_selected = selected_features_model.predict(X_test_eng)

print("--- Model Performance (with Selected Features) ---")
print(classification_report(y_test_eng, y_pred_selected))

## Compare model Performance

In [ ]:
print("--- Baseline Model Performance ---")
print(classification_report(y_test_base, y_pred_base))

print("\n--- Enhanced Model Performance (with Feature Engineering) ---")
print(classification_report(y_test_eng, y_pred_eng))

print("\n--- Model Performance (with Selected Features) ---")
print(classification_report(y_test_eng, y_pred_selected))

# Summarize the performance metrics
print("\n--- Performance Summary ---")
print("Metric         | Baseline | Enhanced | Selected Features")
print("---------------|----------|----------|-------------------")
print(f"Accuracy       | {accuracy_score(y_test_base, y_pred_base):<8.2f} | {accuracy_score(y_test_eng, y_pred_eng):<8.2f} | {accuracy_score(y_test_eng, y_pred_selected):<8.2f}")

# Extract F1-score for class 1 (Churn) from classification reports
report_base = classification_report(y_test_base, y_pred_base, output_dict=True)
report_eng = classification_report(y_test_eng, y_pred_eng, output_dict=True)
report_selected = classification_report(y_test_eng, y_pred_selected, output_dict=True)

f1_churn_base = report_base['1']['f1-score']
f1_churn_eng = report_eng['1']['f1-score']
f1_churn_selected = report_selected['1']['f1-score']

print(f"F1-Score (Churn)| {f1_churn_base:<8.2f} | {f1_churn_eng:<8.2f} | {f1_churn_selected:<8.2f}")

## Model Evaluation Code for Churn Prediction

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Define models to evaluate
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVC (RBF Kernel)": SVC(kernel='rbf', probability=True, random_state=42)
}

results = []

for name, base_model in models.items():
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_eng),
        ('classifier', base_model)
    ])

    # Fit the pipeline
    model_pipeline.fit(X_train_eng, y_train_eng)
    y_pred = model_pipeline.predict(X_test_eng)

    # Evaluate
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test_eng, y_pred),
        'Precision': precision_score(y_test_eng, y_pred),
        'Recall': recall_score(y_test_eng, y_pred),
        'F1-Score': f1_score(y_test_eng, y_pred)
    }
    results.append(metrics)

# Display results
results_df = pd.DataFrame(results)
results_df = results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]
results_df.sort_values(by='F1-Score', ascending=False, inplace=True)

print(results_df)

## SUMMARY

Feature Engineering Techniques

Throughout the project, several advanced feature engineering steps were applied to improve model performance:

Binned Tenure into groups like '0-1 Year', '1-2 Years', etc.

Replaced "No internet service" and "No phone service" with "No" in applicable columns.

Created new interaction features, including:

num_add_services - count of additional services used (e.g., OnlineSecurity, StreamingTV).

monthly_charge_ratio - MonthlyCharges / (tenure + 1)

High_Value_Customer - binary feature indicating high total charges.

TotalCharges_to_MonthlyCharges - a proxy for tenure.

TotalServices - total number of subscribed services.

Age_Group - derived from SeniorCitizen.

Has_Partner_or_Dependents - new binary indicator.

To reduce noise and dimensionality, the following methods were explored:

SelectFromModel using RandomForestClassifier with different thresholds (median, mean).

RFE (Recursive Feature Elimination) using LogisticRegression.

Mutual Information & Chi-squared tests to rank top k features.

Features consistently selected across methods included: Contract, MonthlyCharges, TotalCharges_to_MonthlyCharges, TotalServices, and tenure_group.

## Models Evaluated
  Model Accuracy  Precision    Recall  F1-Score

Logistic Regression  0.806955   0.675862  0.524064  0.590361

Gradient Boosting  0.799858   0.659722  0.508021  0.574018

SVC (RBF Kernel)  0.797019   0.656028  0.494652  0.564024

Random Forest  0.793471   0.642612  0.500000  0.562406

### Conclusion

This project demonstrates the power of iterative feature engineering and model evaluation. Starting from a basic logistic regression, we built and evaluated stronger models using engineered features, robust preprocessing, and advanced ensemble methods — achieving substantial gains in churn prediction performance.